# Minoire: EfficientNet-B3 Fashion Classifier

This notebook trains a multi-head EfficientNet-B3 classifier on the Fashion Product Images dataset.
The model predicts three attributes simultaneously from a single clothing image:

- **Category:** `top`, `bottom`, `shoes`
- **Formality:** `casual`, `formal`
- **Color:** 13 simplified color classes (note: color prediction was later replaced by K-means pixel clustering in the main pipeline)

**Hardware:** Google Colab T4 GPU  
**Dataset:** [Fashion Product Images (Small)](https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small) (44,424 items)  
**Best checkpoint:** Epoch 4: Category 99.5%, Formality 97.0%, Color 79.6%

---

## 1. Setup & GPU Check

In [ ]:
!pip install torch torchvision timm kaggle -q

import os
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
Device: Tesla T4


## 2. Kaggle Authentication & Dataset Download

In [ ]:
os.environ['KAGGLE_USERNAME'] = 'kaggle_username'
os.environ['KAGGLE_KEY'] = 'kaggle_key'
print("Kaggle configured")

Kaggle configured


In [ ]:
!pip install kaggle -q
!kaggle datasets list

ref                                                                   title                                                    size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------------  -------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
rauffauzanrambe/fifa-world-cup-2026-player-performance-dataset        FIFA World Cup 2026 Player Performance Dataset        4154062  2026-06-10 12:58:47.093000          19786        488                1  
harshadapatil31/student-performance-and-study-habits-dataset          Student Performance & Study Habits Dataset              15035  2026-07-06 06:28:52.743000           2940         54                1  
abbas829/ecommerce-sales-dataset                                      Ecommerce sales dataset                                110951  2026-07-03 00:59:03.347000           5318      

In [ ]:
!kaggle datasets download -d paramaggarwal/fashion-product-images-small
!unzip -q fashion-product-images-small.zip -d fashion_data
print("Dataset downloaded and extracted")

Dataset URL: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small
License(s): MIT
100% 565M/565M [00:06<00:00, 91.3MB/s]

Dataset downloaded and extracted


## 3. Dataset Exploration

In [ ]:
import pandas as pd

df = pd.read_csv('fashion_data/styles.csv', on_bad_lines='skip')
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSample:")
print(df.head())
print("\nValue counts for masterCategory:")
print(df['masterCategory'].value_counts())

Shape: (44424, 10)

Columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName']

Sample:
      id gender masterCategory subCategory  articleType baseColour  season  \
0  15970    Men        Apparel     Topwear       Shirts  Navy Blue    Fall   
1  39386    Men        Apparel  Bottomwear        Jeans       Blue  Summer   
2  59263  Women    Accessories     Watches      Watches     Silver  Winter   
3  21379    Men        Apparel  Bottomwear  Track Pants      Black    Fall   
4  53759    Men        Apparel     Topwear      Tshirts       Grey  Summer   

     year   usage                             productDisplayName  
0  2011.0  Casual               Turtle Check Men Navy Blue Shirt  
1  2012.0  Casual             Peter England Men Party Blue Jeans  
2  2016.0  Casual                       Titan Women Silver Watch  
3  2011.0  Casual  Manchester United Men Solid Black Track Pants  
4  2012.0  Casual             

In [ ]:
print("subCategory counts:")
print(df['subCategory'].value_counts().head(20))

print("\nbaseColour counts:")
print(df['baseColour'].value_counts().head(20))

print("\nusage counts (formality):")
print(df['usage'].value_counts())

print("\nseason counts:")
print(df['season'].value_counts())

image_dir = 'fashion_data/images'
image_files = os.listdir(image_dir)
print(f"\nTotal images available: {len(image_files)}")

df['image_path'] = df['id'].apply(lambda x: f"{image_dir}/{x}.jpg")
df['has_image'] = df['image_path'].apply(os.path.exists)
print(f"Items with images: {df['has_image'].sum()}")

subCategory counts:
subCategory
Topwear                     15402
Shoes                        7343
Bags                         3055
Bottomwear                   2694
Watches                      2542
Innerwear                    1808
Jewellery                    1079
Eyewear                      1073
Fragrance                    1011
Sandal                        963
Wallets                       933
Flip Flops                    913
Belts                         811
Socks                         698
Lips                          527
Dress                         478
Loungewear and Nightwear      470
Saree                         427
Nails                         329
Makeup                        307
Name: count, dtype: int64

baseColour counts:
baseColour
Black        9728
White        5538
Blue         4918
Brown        3494
Grey         2741
Red          2455
Green        2115
Pink         1860
Navy Blue    1789
Purple       1640
Silver       1090
Yellow        778
Beige         7

## 4. Label Mapping

The dataset's native labels don't align with minoire's taxonomy. We remap:

- **Category**: subCategory -> `top / bottom / shoes`
- **Formality**: usage -> `casual / formal` (smart_casual and party had < 35 samples each so we merged into formal; Ethnic dropped as it describes cultural style, not formality)
- **Color**: 20+ baseColour values -> 13 simplified classes

In [ ]:
import numpy as np
from PIL import Image

# filter to apparel and footwear only for now
df_filtered = df[df['masterCategory'].isin(['Apparel', 'Footwear'])].copy()
df_filtered = df_filtered[df_filtered['has_image'] == True].copy()
print(f"Filtered dataset size: {len(df_filtered)}")

# map subCategory to Categories
category_map = {
    'Topwear': 'top',
    'Dress': 'top',
    'Bottomwear': 'bottom',
    'Shoes': 'shoes',
    'Sandal': 'shoes',
    'Flip Flops': 'shoes',
    'Socks': 'shoes',
    'Innerwear': 'top',
    'Loungewear and Nightwear': 'top',
}
df_filtered['category_label'] = df_filtered['subCategory'].map(category_map)
df_filtered = df_filtered[df_filtered['category_label'].notna()].copy()

# map usage to formality
formality_map = {
    'Casual': 'casual',
    'Sports': 'casual',
    'Home': 'casual',
    'Travel': 'casual',
    'Smart Casual': 'smart_casual',
    'Formal': 'formal',
    'Party': 'party',
}
df_filtered['formality_label'] = df_filtered['usage'].map(formality_map)

# map colors to simplified palette
color_map = {
    'Black': 'black', 'White': 'white', 'Blue': 'blue',
    'Navy Blue': 'blue', 'Brown': 'brown', 'Grey': 'grey',
    'Red': 'red', 'Green': 'green', 'Pink': 'pink',
    'Purple': 'purple', 'Yellow': 'yellow', 'Orange': 'orange',
    'Beige': 'beige', 'Cream': 'beige', 'Maroon': 'red',
    'Olive': 'green', 'Gold': 'yellow', 'Silver': 'grey',
    'Multi': 'multi',
}
df_filtered['color_label'] = df_filtered['baseColour'].map(color_map)

# drop rows with missing labels
df_filtered = df_filtered.dropna(subset=['category_label', 'formality_label', 'color_label'])

print(f"\nFinal dataset size: {len(df_filtered)}")
print(f"\nCategory distribution:\n{df_filtered['category_label'].value_counts()}")
print(f"\nFormality distribution:\n{df_filtered['formality_label'].value_counts()}")
print(f"\nColor distribution:\n{df_filtered['color_label'].value_counts()}")

Filtered dataset size: 30611

Final dataset size: 26305

Category distribution:
category_label
top       15002
shoes      8913
bottom     2390
Name: count, dtype: int64

Formality distribution:
formality_label
casual          24502
formal           1748
smart_casual       34
party              21
Name: count, dtype: int64

Color distribution:
color_label
black     5890
blue      5116
white     4015
grey      2301
red       1887
brown     1743
green     1552
pink      1070
purple     954
yellow     760
beige      590
orange     279
multi      148
Name: count, dtype: int64


there's a huge imbalance between the top (15k) and bottom (2.3k)

In [ ]:
print(df[df['masterCategory'] == 'Apparel']['subCategory'].value_counts())

subCategory
Topwear                     15402
Bottomwear                   2694
Innerwear                    1808
Dress                         478
Loungewear and Nightwear      470
Saree                         427
Apparel Set                   106
Socks                          12
Name: count, dtype: int64


In [ ]:
# Add Apparel Set to top
category_map['Apparel Set'] = 'top'

# Rebuild filtered dataset with updated map
df_filtered['category_label'] = df_filtered['subCategory'].map(category_map)
df_filtered = df_filtered[df_filtered['category_label'].notna()].copy()

print("Updated category distribution:")
print(df_filtered['category_label'].value_counts())
print(f"\nTotal samples: {len(df_filtered)}")

Updated category distribution:
category_label
top       15002
shoes      8913
bottom     2390
Name: count, dtype: int64

Total samples: 26305


## 5. Label Encoding & Class Imbalance

Two significant imbalances:
- **Category**: bottom (2,390) vs top (15,002); bottom oversampled 4x
- **Formality**: casual (24,502) vs formal (1,803); handled via weighted CrossEntropyLoss

In [ ]:
# we'll merge smart_casual and party due to insufficient samples
df_filtered['formality_label'] = df_filtered['formality_label'].replace({
    'smart_casual': 'formal',
    'party': 'formal'
})

print("Revised formality distribution:")
print(df_filtered['formality_label'].value_counts())

from sklearn.preprocessing import LabelEncoder

category_encoder = LabelEncoder()
formality_encoder = LabelEncoder()
color_encoder = LabelEncoder()

df_filtered['category_idx'] = category_encoder.fit_transform(df_filtered['category_label'])
df_filtered['formality_idx'] = formality_encoder.fit_transform(df_filtered['formality_label'])
df_filtered['color_idx'] = color_encoder.fit_transform(df_filtered['color_label'])

print("\nCategory classes:", category_encoder.classes_.tolist())
print("Formality classes:", formality_encoder.classes_.tolist())
print("Color classes:", color_encoder.classes_.tolist())

# save label encoders for inference
import pickle
with open('category_encoder.pkl', 'wb') as f:
    pickle.dump(category_encoder, f)
with open('formality_encoder.pkl', 'wb') as f:
    pickle.dump(formality_encoder, f)
with open('color_encoder.pkl', 'wb') as f:
    pickle.dump(color_encoder, f)

print("\nLabel encoders saved")

Revised formality distribution:
formality_label
casual    24502
formal     1803
Name: count, dtype: int64

Category classes: ['bottom', 'shoes', 'top']
Formality classes: ['casual', 'formal']
Color classes: ['beige', 'black', 'blue', 'brown', 'green', 'grey', 'multi', 'orange', 'pink', 'purple', 'red', 'white', 'yellow']

Label encoders saved


## 6. Dataset & DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

class FashionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return {
            'image': image,
            'category': torch.tensor(row['category_idx'], dtype=torch.long),
            'formality': torch.tensor(row['formality_idx'], dtype=torch.long),
            'color': torch.tensor(row['color_idx'], dtype=torch.long),
        }

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

augment_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

df_bottom = df_filtered[df_filtered['category_label'] == 'bottom']
df_bottom_aug = pd.concat([df_bottom] * 4, ignore_index=True)  # 4x oversample

from sklearn.model_selection import train_test_split

df_train_base, df_val = train_test_split(df_filtered, test_size=0.15, random_state=42, stratify=df_filtered['category_label'])
df_train = pd.concat([df_train_base, df_bottom_aug], ignore_index=True).sample(frac=1, random_state=42)

print(f"Train size: {len(df_train)}")
print(f"Val size: {len(df_val)}")
print(f"\nTrain category distribution:")
print(df_train['category_label'].value_counts())

train_dataset = FashionDataset(df_train, transform=augment_transform)
val_dataset = FashionDataset(df_val, transform=base_transform)

category_counts = df_train['category_label'].value_counts()
category_weights = torch.tensor([
    1.0 / category_counts['bottom'],
    1.0 / category_counts['shoes'],
    1.0 / category_counts['top']
], dtype=torch.float32)
category_weights = category_weights / category_weights.sum()

formality_counts = df_train['formality_label'].value_counts()
formality_weights = torch.tensor([
    1.0 / formality_counts['casual'],
    1.0 / formality_counts['formal'],
], dtype=torch.float32)
formality_weights = formality_weights / formality_weights.sum()

print("\nCategory weights:", category_weights)
print("Formality weights:", formality_weights)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print("\nDataloaders ready")

Train size: 31919
Val size: 3946

Train category distribution:
category_label
top       12752
bottom    11591
shoes      7576
Name: count, dtype: int64

Category weights: tensor([0.2908, 0.4449, 0.2643])
Formality weights: tensor([0.0764, 0.9236])

Dataloaders ready


## 7. Model Architecture

EfficientNet-B3 backbone with three independent classification heads sharing the same 1,536-dimensional feature vector.

- **Backbone**: pretrained ImageNet weights via timm, lr = 1e-4 (lower to preserve features)
- **Heads**: trained from scratch, lr = 3e-4
- **Combined loss**: `0.4 * category + 0.3 * formality + 0.3 * color`

In [ ]:
import torch
import torch.nn as nn
import timm

class FashionClassifier(nn.Module):
    def __init__(self, num_categories, num_formalities, num_colors):
        super().__init__()

        self.backbone = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
        backbone_features = self.backbone.num_features  # 1536 for B3

        # Shared dropout
        self.dropout = nn.Dropout(0.3)

        # Three separate classification heads
        self.category_head = nn.Sequential(
            nn.Linear(backbone_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_categories)
        )
        self.formality_head = nn.Sequential(
            nn.Linear(backbone_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_formalities)
        )
        self.color_head = nn.Sequential(
            nn.Linear(backbone_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_colors)
        )

    def forward(self, x):
        features = self.backbone(x)
        features = self.dropout(features)
        return {
            'category': self.category_head(features),
            'formality': self.formality_head(features),
            'color': self.color_head(features)
        }

# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = FashionClassifier(
    num_categories=len(category_encoder.classes_),
    num_formalities=len(formality_encoder.classes_),
    num_colors=len(color_encoder.classes_)
).to(device)

# Loss functions with class weights
category_criterion = nn.CrossEntropyLoss(weight=category_weights.to(device))
formality_criterion = nn.CrossEntropyLoss(weight=formality_weights.to(device))
color_criterion = nn.CrossEntropyLoss()

# Optimizer: lower lr for backbone, higher for heads
optimizer = torch.optim.AdamW([
    {'params': model.backbone.parameters(), 'lr': 1e-4},
    {'params': model.category_head.parameters(), 'lr': 3e-4},
    {'params': model.formality_head.parameters(), 'lr': 3e-4},
    {'params': model.color_head.parameters(), 'lr': 3e-4},
], weight_decay=1e-4)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print(f"Model initialized on {device}")
print(f"Backbone features: {model.backbone.num_features}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

model.safetensors: reconstructing file:   0%|          |  0.00B / 49.3MB            

model.safetensors: downloading bytes:           |  0.00B            

Model initialized on cuda
Backbone features: 1536
Total parameters: 11,881,274
Trainable parameters: 11,881,274


## 8. Training Loop

Training stopped at epoch 6. Best checkpoint saved at epoch 4.

| Epoch | Val Loss | Category | Formality | Color |
|-------|----------|----------|-----------|-------|
| 1 | 0.2934 | 99.4% | 92.8% | 76.2% |
| 2 | 0.2655 | 99.3% | 93.2% | 77.6% |
| 3 | 0.2505 | 99.5% | 95.0% | 79.1% |
| **4** | **0.2394** | **99.5%** | **97.0%** | **79.6%** |
| 5 | 0.2471 | 99.5% | 97.0% | 79.7% |
| 6 | 0.2514 | 99.7% | 97.1% | 80.4% |

In [ ]:
def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    cat_correct = form_correct = color_correct = total = 0

    for batch in loader:
        images = batch['image'].to(device)
        cat_labels = batch['category'].to(device)
        form_labels = batch['formality'].to(device)
        color_labels = batch['color'].to(device)

        optimizer.zero_grad()
        outputs = model(images)

        cat_loss = category_criterion(outputs['category'], cat_labels)
        form_loss = formality_criterion(outputs['formality'], form_labels)
        color_loss = color_criterion(outputs['color'], color_labels)

        # Combined loss — category matters most
        loss = 0.4 * cat_loss + 0.3 * form_loss + 0.3 * color_loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total += images.size(0)

        cat_correct += (outputs['category'].argmax(1) == cat_labels).sum().item()
        form_correct += (outputs['formality'].argmax(1) == form_labels).sum().item()
        color_correct += (outputs['color'].argmax(1) == color_labels).sum().item()

    n = len(loader)
    return {
        'loss': total_loss / n,
        'cat_acc': cat_correct / total,
        'form_acc': form_correct / total,
        'color_acc': color_correct / total,
    }

def val_epoch(model, loader, device):
    model.eval()
    total_loss = 0
    cat_correct = form_correct = color_correct = total = 0

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            cat_labels = batch['category'].to(device)
            form_labels = batch['formality'].to(device)
            color_labels = batch['color'].to(device)

            outputs = model(images)

            cat_loss = category_criterion(outputs['category'], cat_labels)
            form_loss = formality_criterion(outputs['formality'], form_labels)
            color_loss = color_criterion(outputs['color'], color_labels)

            loss = 0.4 * cat_loss + 0.3 * form_loss + 0.3 * color_loss
            total_loss += loss.item()
            total += images.size(0)

            cat_correct += (outputs['category'].argmax(1) == cat_labels).sum().item()
            form_correct += (outputs['formality'].argmax(1) == form_labels).sum().item()
            color_correct += (outputs['color'].argmax(1) == color_labels).sum().item()

    n = len(loader)
    return {
        'loss': total_loss / n,
        'cat_acc': cat_correct / total,
        'form_acc': form_correct / total,
        'color_acc': color_correct / total,
    }

# Training loop
EPOCHS = 10
best_val_loss = float('inf')
best_model_path = 'minoire_fashion_classifier.pth'

print("Starting training...\n")

for epoch in range(EPOCHS):
    train_metrics = train_epoch(model, train_loader, optimizer, device)
    val_metrics = val_epoch(model, val_loader, device)
    scheduler.step()

    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"  Train — Loss: {train_metrics['loss']:.4f} | Cat: {train_metrics['cat_acc']:.3f} | Form: {train_metrics['form_acc']:.3f} | Color: {train_metrics['color_acc']:.3f}")
    print(f"  Val   — Loss: {val_metrics['loss']:.4f} | Cat: {val_metrics['cat_acc']:.3f} | Form: {val_metrics['form_acc']:.3f} | Color: {val_metrics['color_acc']:.3f}")

    if val_metrics['loss'] < best_val_loss:
        best_val_loss = val_metrics['loss']
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': best_val_loss,
            'category_classes': category_encoder.classes_.tolist(),
            'formality_classes': formality_encoder.classes_.tolist(),
            'color_classes': color_encoder.classes_.tolist(),
        }, best_model_path)
        print(f"  ✓ Best model saved (val_loss: {best_val_loss:.4f})")
    print()

print("Training complete.")
print(f"Best val loss: {best_val_loss:.4f}")

## 9. Download Trained Model Weights

In [ ]:
from google.colab import files

files.download('minoire_fashion_classifier.pth')
files.download('category_encoder.pkl')
files.download('formality_encoder.pkl')
files.download('color_encoder.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Notes

**Color head**: The color classification head is trained here but is **not used in the production pipeline**. Color prediction was replaced by K-means clustering in Lab color space directly on pixel values (`app/services/cv.py`), which is more accurate and not dependent on training data distribution.

**Formality**: The model outputs binary casual/formal. The original four-tier taxonomy (casual, smart_casual, formal, party) was not achievable with this dataset due to insufficient samples for smart_casual and party. Retraining with a better-labeled dataset is planned.

**Reproducibility**: Set `random_state=42` throughout. Results may vary slightly across GPU runs due to non-deterministic CUDA operations.